# 固定効果モデル (project_start_year, country, evaluator)

固定効果として project_start_year / country / evaluator を入れたOLSを推定する。


In [246]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

print('ライブラリのインポート完了')


ライブラリのインポート完了


In [247]:
# データ読み込み
df = pd.read_csv('../df_check_99.csv', index_col=0)
print(f'データ形状: {df.shape}')
print(f'列数: {len(df.columns)}')


データ形状: (2227, 78)
列数: 78


In [248]:
# 対数変換する変数
cols_log = [
    'project_cost_plan',
    'project_cost_act',
    'project_duration_act',
    'project_duration_plan',
    'population'
]

for col in cols_log:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: np.log(x) if pd.notna(x) and x > 0 else np.nan)
        print(f'{col}: 対数変換完了')

print('対数変換完了')


project_cost_plan: 対数変換完了
project_cost_act: 対数変換完了
project_duration_act: 対数変換完了
project_duration_plan: 対数変換完了
population: 対数変換完了
対数変換完了


In [249]:
# セクターフラグ変数
cat_flg = [col for col in df.columns if col.endswith('_flg_fix')]
print(f'セクターフラグ変数数: {len(cat_flg)}')

# その他の説明変数
others = [
    'project_cost_plan',
    'project_duration_plan',
    'gdp_growth',
    'population',
    'political_stability',
    'regulatory_quality',
    'freedom_house_score',
    'voice_and_accountability',
    'control_of_corruption',
    'government_effectiveness',
    'rule_of_law'
]

others = [col for col in others if col in df.columns]
print(f'その他の説明変数数: {len(others)}')

X_vars = others + cat_flg
print(f'説明変数数: {len(X_vars)}')


セクターフラグ変数数: 28
その他の説明変数数: 11
説明変数数: 39


In [250]:
def escape_var_name(var):
    has_japanese = any('぀' <= c <= 'ゟ' or '゠' <= c <= 'ヿ' or '一' <= c <= '鿿' for c in var)
    if ' ' in var or '-' in var or '・' in var or has_japanese:
        return f"Q('{var}')"
    return var

formula_parts = [escape_var_name(var) for var in X_vars]
formula_rhs = ' + '.join(formula_parts) if formula_parts else '1'

cat_terms = []
for c in ['region_detail', 'eval_year', 'external_eval_flg']:
    if c in df.columns:
        cat_terms.append(f'C({c})')

if cat_terms:
    base_formula = f"total_eval ~ {formula_rhs} + " + ' + '.join(cat_terms)
else:
    base_formula = f"total_eval ~ {formula_rhs}"

print('ベース式（先頭100文字）:')
print(base_formula[:100] + ('...' if len(base_formula) > 100 else ''))


ベース式（先頭100文字）:
total_eval ~ project_cost_plan + project_duration_plan + gdp_growth + population + political_stabili...


In [251]:
# 固定効果モデル
required_fe = ['project_start_year', 'country', 'evaluator']
missing_fe = [c for c in required_fe if c not in df.columns]
if missing_fe:
    raise ValueError(f'Missing required columns for fixed effects: {missing_fe}')

df_fe = df.copy()
for c in required_fe:
    df_fe[c] = df_fe[c].astype(str)

analysis_vars = ['total_eval'] + X_vars + required_fe
analysis_vars += [c for c in ['region_detail', 'eval_year', 'external_eval_flg'] if c in df_fe.columns]
analysis_vars = [v for v in analysis_vars if v in df_fe.columns]

df_fe_analysis = df_fe[analysis_vars].dropna()
print(f'固定効果モデル用データ行数: {len(df_fe_analysis)}')

formula_fe = base_formula + ' + C(project_start_year) + C(country) + C(evaluator)'
print('固定効果モデル式（先頭100文字）:')
print(formula_fe[:100] + ('...' if len(formula_fe) > 100 else ''))

model_fe = smf.ols(formula_fe, data=df_fe_analysis).fit()
print(model_fe.summary())


固定効果モデル用データ行数: 2065
固定効果モデル式（先頭100文字）:
total_eval ~ project_cost_plan + project_duration_plan + gdp_growth + population + political_stabili...
                            OLS Regression Results                            
Dep. Variable:             total_eval   R-squared:                       0.548
Model:                            OLS   Adj. R-squared:                  0.250
Method:                 Least Squares   F-statistic:                     1.835
Date:                Sat, 03 Jan 2026   Prob (F-statistic):           7.84e-13
Time:                        14:37:44   Log-Likelihood:                -925.91
No. Observations:                1103   AIC:                             2732.
Df Residuals:                     663   BIC:                             4934.
Df Model:                         439                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          

## 固定効果（吸収）版の推定

ダミーを明示せずに、country / project_start_year / evaluator を固定効果として吸収した推定。


In [252]:
from linearmodels.iv import AbsorbingLS
import patsy

required_fe = ['project_start_year', 'country', 'evaluator']
missing_fe = [c for c in required_fe if c not in df.columns]
if missing_fe:
    raise ValueError(f'Missing required columns for fixed effects: {missing_fe}')

df_abs = df.copy()
df_abs['project_start_year_num'] = pd.to_numeric(df_abs['project_start_year'], errors='coerce')
df_abs['country'] = df_abs['country'].astype(str)
df_abs['evaluator'] = df_abs['evaluator'].astype(str)

analysis_vars = ['total_eval'] + X_vars + ['country', 'evaluator', 'project_start_year_num']
analysis_vars += [c for c in ['region_detail', 'eval_year', 'external_eval_flg'] if c in df_abs.columns]
analysis_vars = [v for v in analysis_vars if v in df_abs.columns]

df_abs = df_abs[analysis_vars].dropna()

# y, X を作成（固定効果は absorb で吸収）
y, X = patsy.dmatrices(base_formula, data=df_abs, return_type='dataframe')
absorb = df_abs[['country', 'project_start_year_num', 'evaluator']]

print('固定効果（吸収）モデル式（先頭100文字）:')
print(base_formula[:100] + ('...' if len(base_formula) > 100 else ''))

model_abs = AbsorbingLS(y, X, absorb=absorb)
res_abs = model_abs.fit(cov_type='clustered', clusters=df_abs['country'])
print(res_abs.summary)


固定効果（吸収）モデル式（先頭100文字）:
total_eval ~ project_cost_plan + project_duration_plan + gdp_growth + population + political_stabili...


ValueError: Cannot cast object dtype to float64